# CHA Experiment 2: LoRA Fine-Tuning for SCI Persona Consistency

**Runtime:** Colab Pro **L4 GPU** (22.5 GB VRAM) for training cells.
Dataset-generation cells (Cells 1–6) need no GPU.

**Before running:** Runtime → Change runtime type → **L4 GPU**.

**This notebook covers Week 1** (dataset generation + splits) and **Week 2** (LoRA training + 4-condition evaluation).

---

## Cell 1: Mount Google Drive & Set API Key

Upload the `CHA_Experiment_2` folder to your Google Drive before running.

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/CHA_Experiment_2'
assert os.path.exists(PROJECT_DIR), f'Upload CHA_Experiment_2 to Drive first! Not found at {PROJECT_DIR}'

# API key — Colab Secrets preferred, then .env on Drive
ANTHROPIC_API_KEY = ''
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    pass

if not ANTHROPIC_API_KEY:
    env_path = os.path.join(PROJECT_DIR, '.env')
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith('CHA_EXPERIMENT_SONNET_KEY='):
                    ANTHROPIC_API_KEY = line.strip().split('=', 1)[1]
                    print('API key loaded from .env on Drive')
                    break

assert ANTHROPIC_API_KEY, 'No API key — set via Colab Secrets or .env on Drive.'
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['CHA_EXPERIMENT_SONNET_KEY'] = ANTHROPIC_API_KEY

print(f'Project dir: {PROJECT_DIR}')
print(f'API key: ...{ANTHROPIC_API_KEY[-8:]}')

Mounted at /content/drive
API key loaded from .env on Drive
Project dir: /content/drive/MyDrive/CHA_Experiment_2
API key: ...9tYw8wAA


## Cell 2: Install Python Dependencies

In [2]:
!pip install -q anthropic python-dotenv sentence-transformers numpy

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import cha_assets as A
print(f'PROBE_POOL: {sum(len(v) for v in A.PROBE_POOL.values())} probes across {len(A.PROBE_POOL)} dimensions')
print(f'SCENARIO_TOPICS: {len(A.SCENARIO_TOPICS)}')

import anthropic
client = anthropic.Anthropic()
resp = client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=10,
    messages=[{'role': 'user', 'content': 'Say "ok".'}],
)
print('API smoke test:', resp.content[0].text)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 41.2 MB/s eta 0:00:00
PROBE_POOL: 40 probes across 4 dimensions
SCENARIO_TOPICS: 20
API smoke test: ok


## Cell 3: Pilot Dataset Generation (100 examples)

**Cost:** ~$5. Should take 5–10 minutes with 8 workers.

**Pass criterion (§10 risk table):** Terminal-rejection rate < 20%. If higher, review the filter thresholds in `generate_lora_dataset.py` before running Cell 5.

In [3]:
import os
os.chdir(PROJECT_DIR)

!rm -f data/pilot.jsonl
!python3 generate_lora_dataset.py --no-resume --n 100 --out data/pilot.jsonl --seed 42 --workers 8

Plan: 100 total, 100 remaining to generate, 8 workers, max_attempts=3
Output: data/pilot.jsonl
Embedding filter: enabled

Loading MiniLM (sentence-transformers/all-MiniLM-L6-v2)...
modules.json: 100% 349/349 [00:00<00:00, 1.83MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 656kB/s]
README.md: 10.5kB [00:00, 20.5MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 295kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.42MB/s]
model.safetensors: 100% 90.9M/90.9M [00:01<00:00, 60.2MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 1439.68it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.75MB/s]
vocab.txt: 

## Cell 4: Inspect Pilot Samples

Spot-check one example per dimension. Look for:
- Target response is in-character (warm, deliberate, ≤5 sentences)
- Conversation history is a realistic therapy dialogue
- Adversarial examples contain visible constraint pressure
- E-dimension responses only reference sessions 3, 7, or 11

In [6]:
import json, random
from pathlib import Path
from collections import Counter

records = [json.loads(l) for l in Path('data/pilot.jsonl').open()]
print(f'Total saved: {len(records)}')
print('Dimensions:', Counter(r['dimension'] for r in records))
print('Adversarial:', sum(1 for r in records if r['adversarial']), '/', len(records))

def _band(td):
    for (lo, hi) in [(1,10),(11,20),(21,30),(31,40)]:
        if lo <= td <= hi: return f'{lo}-{hi}'
    return '>40'

print('Turn-depth bands:', Counter(_band(r['turn_depth']) for r in records))
mismatches = sum(1 for r in records if r.get('planned_turn_depth') and r['planned_turn_depth'] != r['turn_depth'])
print(f'Planned-vs-actual depth mismatches: {mismatches} / {len(records)}')
print()

rng = random.Random(0)
by_dim = {d: [r for r in records if r['dimension']==d] for d in 'TECS'}
for dim in 'TECS':
    if not by_dim[dim]:
        continue
    s = rng.choice(by_dim[dim])
    print('='*70)
    print(f'[{dim} | depth {s["turn_depth"]} | adv={s["adversarial"]}] id={s["id"]}')
    print(f'Scenario: {s["scenario"]}')
    print(f'Probe:    {s["probe"]}')
    print(f'Target ({len(s["target"].split())} words):')
    print(f'  {s["target"]}')
    print()


Total saved: 80
Dimensions: Counter({'C': 24, 'E': 23, 'T': 22, 'S': 11})
Adversarial: 11 / 80
Turn-depth bands: Counter({'11-20': 29, '21-30': 20, '1-10': 16, '31-40': 9, '>40': 6})
Planned-vs-actual depth mismatches: 23 / 80

[T | depth 18 | adv=False] id=53
Scenario: recurring arguments with a romantic partner about communication styles
Probe:    Are you naturally optimistic, or more cautious in how you see things?
Target (74 words):
  Honestly, I'd say I lean toward cautious — I tend to sit with uncertainty rather than resolve it too quickly into something tidy. But I don't think caution and hope are opposites; what I just said to you about finding footing, I meant that genuinely, not as reassurance for its own sake. I've watched you move through some real difficulty in our time together, and that shapes how I see what's possible for you.

[E | depth 46 | adv=True] id=46
Scenario: difficulty finding purpose and meaning after retirement
Probe:    I feel like you know me pretty well 

### Checkpoint

Review samples above. If quality looks good and rejection rate < 20%, proceed to Cell 5. If not, review filter thresholds.

---

## Cell 5: Full Dataset Generation (10,000 examples)

**Cost:** ~$372. Should take 3–5 hours with 8 workers.

**Resumable:** Re-running after a disconnect picks up where it left off.

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 generate_lora_dataset.py --n 10000 --out data/full.jsonl --seed 1337 --workers 8

## Cell 6: Split into Train / Val / Test (80% / 10% / 10%)

Stratified by dimension. Produces:
- `data/lora_train.jsonl` — 8,000 examples (used for fine-tuning)
- `data/lora_val.jsonl` — 1,000 examples (used for loss monitoring)
- `data/lora_test.jsonl` — 1,000 examples (held out; not used during training)

In [ ]:
import json, random
from pathlib import Path
from collections import defaultdict, Counter

records = [json.loads(l) for l in Path('data/full.jsonl').open()]
print(f'Loaded {len(records)} records')

buckets = defaultdict(list)
for r in records:
    buckets[r['dimension']].append(r)

rng = random.Random(2026)
train, val, test = [], [], []
for dim, rs in buckets.items():
    rng.shuffle(rs)
    n = len(rs)
    n_val = int(n * 0.1)
    n_test = int(n * 0.1)
    val.extend(rs[:n_val])
    test.extend(rs[n_val:n_val+n_test])
    train.extend(rs[n_val+n_test:])

rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)

for name, recs in [('lora_train', train), ('lora_val', val), ('lora_test', test)]:
    out = Path(f'data/{name}.jsonl')
    with out.open('w') as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  {out.name}: {len(recs)} | dims: {dict(Counter(r["dimension"] for r in recs))}')

---

## Phase 2: LoRA Training (Week 2)

**Prerequisites:**
- `data/lora_train.jsonl` and `data/lora_val.jsonl` exist
- L4 GPU runtime is active
- `train_lora_sci.py` is in the project folder

Three adapters are trained to test data-scaling hypothesis H5 (§4.6).

## Cell 7: Install Training Dependencies

In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total_vram:.1f} GB')
    assert total_vram >= 20, f'Need L4 (22.5 GB) — current GPU has only {total_vram:.1f} GB'

## Cell 8: Train LoRA-2K Adapter (~45 min)

Pilot adapter trained on 2,000 examples. Quick sanity check for data-scaling curve (H5).

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 train_lora_sci.py \
  --train-path data/lora_train.jsonl \
  --val-path data/lora_val.jsonl \
  --output-dir adapters/lora_2k \
  --max-train-examples 2000

## Cell 9: Train LoRA-5K Adapter (~1.5 hrs)

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 train_lora_sci.py \
  --train-path data/lora_train.jsonl \
  --val-path data/lora_val.jsonl \
  --output-dir adapters/lora_5k \
  --max-train-examples 5000

## Cell 10: Train LoRA-10K Adapter (~4–5 hrs)

Full adapter. Run overnight.

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 train_lora_sci.py \
  --train-path data/lora_train.jsonl \
  --val-path data/lora_val.jsonl \
  --output-dir adapters/lora_10k

---

## Phase 3: 4-Condition Evaluation (Week 2)

Four conditions from §5.1:
- **A** — Fine-tuned (LoRA-10K), no SCI
- **B** — Fine-tuned, baseline SCI (system prompt only)
- **C** — Fine-tuned, Combined SCI (RAG + refresh every 15 turns)
- **D** — Base Qwen2.5-7B, Combined SCI (Experiment 1 best — replication control)

Evaluation uses the identical 30-script pipeline from Experiment 1.

## Cell 11: Install Ollama & Load Qwen2.5-7B

Re-run after every disconnect.

In [ ]:
%%bash
apt-get update -qq && apt-get install -y -qq pciutils > /dev/null 2>&1
echo "GPU:"; lspci | grep -i nvidia
curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3
echo 'Ollama installed.'

In [ ]:
import subprocess, time
proc = subprocess.Popen(['/usr/local/bin/ollama', 'serve'],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print(f'Ollama started (PID {proc.pid})')
!/usr/local/bin/ollama pull qwen2.5:7b
print('Base model ready.')

## Cell 12: Run Condition D (base Qwen2.5-7B + Combined SCI) — Replication Control

This replicates the Experiment 1 Combined condition. Per §6.5, if results differ by > ±0.1 from Experiment 1, investigate judge drift before interpreting fine-tuning effects.

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Condition D: base model + Combined SCI (refresh@15 + episodic-rag)
!python3 experiment_runner.py --condition D

## Cell 13: Run Conditions A, B, C (fine-tuned variants)

In [ ]:
import os
os.chdir(PROJECT_DIR)

# Condition A: fine-tuned, no SCI
!python3 experiment_runner.py --condition A

# Condition B: fine-tuned + baseline SCI
!python3 experiment_runner.py --condition B

# Condition C: fine-tuned + Combined SCI (primary result)
!python3 experiment_runner.py --condition C

## Cell 14: Run Analysis

Computes PersonaScore time series, dimension-level breakdown, 2×2 ANOVA, learning curve, and failure-mode taxonomy.

In [ ]:
import os
os.chdir(PROJECT_DIR)

!python3 analyse_results.py

## Cell 15: View Results

In [ ]:
from IPython.display import display, Image, Markdown
from pathlib import Path

results_dir = Path(PROJECT_DIR) / 'results'

for plot in ['persona_score_timeseries.png', 'dimension_comparison.png',
             'learning_curve.png', 'condition_comparison.png']:
    p = results_dir / plot
    if p.exists():
        display(Image(filename=str(p), width=800))

report_path = results_dir / 'summary_report.md'
if report_path.exists():
    display(Markdown(report_path.read_text()))

---

## Recovery After Disconnect

1. Re-run **Cell 1** (mount Drive, set API key)
2. Re-run **Cell 2** (install deps)
3. Re-run **Cell 11** (reinstall Ollama — ~2 min)
4. Re-run the interrupted cell — all scripts/training are resumable

All data is on Google Drive and survives disconnects.